In [ ]:
import os

from dotenv import load_dotenv

load_dotenv(os.path.join("..", ".env"), override=True)

%load_ext autoreload
%autoreload 2

# 用于研究的深度智能体

## 概述

<img src="./assets/agent_header.png" width="800" style="display:block; margin-left:0;">

现在，我们可以将我们学到的一切整合在一起：

* 我们将使用 **TODOs** 来跟踪任务。
* 我们将使用 **文件** 来存储原始工具调用结果。
* 我们将**将研究任务委托给子智能体**以实现上下文隔离。

## 搜索工具

我们将构建一个搜索工具，将原始内容卸载到文件中，只向智能体返回摘要。这是长期运行智能体轨迹的常见模式，[正如我们在 Manus 中看到的那样](https://manus.im/blog/Context-Engineering-for-AI-Agents-Lessons-from-Building-Manus)！

### 核心组件

1. **搜索执行 (`run_tavily_search`)**：使用 Tavily API 执行实际的网络搜索，具有可配置的结果数量和主题过滤参数。

2. **内容摘要 (`summarize_webpage_content`)**：使用轻量级模型（GPT-4o-mini）生成网页内容的结构化摘要，生成描述性文件名和关键学习摘要。

3. **结果处理 (`process_search_results`)**：通过 HTTP 获取完整网页内容，使用 `markdownify` 将 HTML 转换为 markdown，并为每个结果生成摘要。

4. **上下文卸载 (`tavily_search` 工具)**：主要工具，它：
   - 执行搜索并处理结果
   - 将完整内容保存到智能体状态中的文件（上下文卸载）
   - 只向智能体返回最小摘要（防止上下文垃圾信息）
   - 使用 LangGraph `Command` 更新文件和消息

5. **战略思考 (`think_tool`)**：为智能体提供结构化反思机制，分析发现、评估差距并规划研究工作流程中的下一步。

这种架构通过将详细搜索结果存储在文件中，同时保持智能体的工作上下文最小化和专注，解决了令牌效率问题。

In [ ]:
%%writefile ../src/deep_agents_from_scratch/research_tools.py
"""研究工具。

此模块为研究智能体提供搜索和内容处理实用程序，
包括网络搜索功能和内容摘要工具。
"""
import os
from datetime import datetime
import uuid, base64

import httpx
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import InjectedToolArg, InjectedToolCallId, tool
from langgraph.prebuilt import InjectedState
from langgraph.types import Command
from markdownify import markdownify
from pydantic import BaseModel, Field
from tavily import TavilyClient
from typing_extensions import Annotated, Literal

from deep_agents_from_scratch.prompts import SUMMARIZE_WEB_SEARCH
from deep_agents_from_scratch.state import DeepAgentState

# 摘要模型
summarization_model = init_chat_model(model="openai:gpt-4o-mini")
tavily_client = TavilyClient()

class Summary(BaseModel):
    """网页内容摘要的模式。"""
    filename: str = Field(description="要存储的文件名。")
    summary: str = Field(description="从网页中获得的关键学习。")

def get_today_str() -> str:
    """获取人类可读格式的当前日期。"""
    return datetime.now().strftime("%a %b %-d, %Y")

def run_tavily_search(
    search_query: str, 
    max_results: int = 1, 
    topic: Literal["general", "news", "finance"] = "general", 
    include_raw_content: bool = True, 
) -> dict:
    """使用 Tavily API 对单个查询执行搜索。

    Args:
        search_query: 要执行的搜索查询
        max_results: 每个查询的最大结果数
        topic: 搜索结果的主题过滤器
        include_raw_content: 是否包含原始网页内容

    Returns:
        搜索结果字典
    """
    result = tavily_client.search(
        search_query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic
    )

    return result

def summarize_webpage_content(webpage_content: str) -> Summary:
    """使用配置的摘要模型摘要网页内容。
    
    Args:
        webpage_content: 要摘要的原始网页内容
        
    Returns:
        包含文件名和摘要的 Summary 对象
    """
    try:
        # 设置结构化输出模型进行摘要
        structured_model = summarization_model.with_structured_output(Summary)
        
        # 生成摘要
        summary_and_filename = structured_model.invoke([
            HumanMessage(content=SUMMARIZE_WEB_SEARCH.format(
                webpage_content=webpage_content, 
                date=get_today_str()
            ))
        ])
        
        return summary_and_filename
        
    except Exception:
        # 失败时返回基本摘要对象
        return Summary(
            filename="search_result.md",
            summary=webpage_content[:1000] + "..." if len(webpage_content) > 1000 else webpage_content
        )

def process_search_results(results: dict) -> list[dict]:
    """通过摘要可用内容来处理搜索结果。
    
    Args:
        results: Tavily 搜索结果字典
        
    Returns:
        带有摘要的已处理结果列表
    """
    processed_results = []

    # 创建 HTTP 请求客户端
    HTTPX_CLIENT = httpx.Client()
    
    for result in results.get('results', []):
        
        # 获取 url
        url = result['url']
        
        # 读取 url
        response = HTTPX_CLIENT.get(url)

        if response.status_code == 200:
            # 将 HTML 转换为 markdown
            raw_content = markdownify(response.text)
            summary_obj = summarize_webpage_content(raw_content)
        else:
            # 使用 Tavily 生成的摘要
            raw_content = result.get('raw_content', '')
            summary_obj = Summary(
                filename="URL_error.md",
                summary=result.get('content', '读取 URL 时出错；请尝试其他搜索。')
            )
        
        # 唯一化文件名
        uid = base64.urlsafe_b64encode(uuid.uuid4().bytes).rstrip(b"=").decode("ascii")[:8]
        name, ext = os.path.splitext(summary_obj.filename)
        summary_obj.filename = f"{name}_{uid}{ext}"

        processed_results.append({
            'url': result['url'],
            'title': result['title'],
            'summary': summary_obj.summary,
            'filename': summary_obj.filename,
            'raw_content': raw_content,
        })
    
    return processed_results

@tool(parse_docstring=True)
def tavily_search(
    query: str,
    state: Annotated[DeepAgentState, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
    max_results: Annotated[int, InjectedToolArg] = 1,
    topic: Annotated[Literal["general", "news", "finance"], InjectedToolArg] = "general",
) -> Command:
    """搜索网络并将详细结果保存到文件，同时返回最小上下文。

    执行网络搜索并将完整内容保存到文件以进行上下文卸载。
    只返回必要信息以帮助智能体决定下一步。

    Args:
        query: 要执行的搜索查询
        state: 用于文件存储的注入智能体状态
        tool_call_id: 注入的工具调用标识符
        max_results: 要返回的最大结果数（默认：1）
        topic: 主题过滤器 - 'general'、'news' 或 'finance'（默认：'general'）

    Returns:
        将完整结果保存到文件并提供最小摘要的命令
    """
    # 执行搜索
    search_results = run_tavily_search(
        query,
        max_results=max_results,
        topic=topic,
        include_raw_content=True,
    ) 

    # 处理和摘要结果
    processed_results = process_search_results(search_results)
    
    # 将每个结果保存到文件并准备摘要
    files = state.get("files", {})
    saved_files = []
    summaries = []
    
    for i, result in enumerate(processed_results):
        # 使用摘要生成的 AI 文件名
        filename = result['filename']
        
        # 创建包含完整详细信息的文件内容
        file_content = f"""# 搜索结果：{result['title']}

**URL：** {result['url']}
**查询：** {query}
**日期：** {get_today_str()}

## 摘要
{result['summary']}

## 原始内容
{result['raw_content'] if result['raw_content'] else '无原始内容可用'}
"""
        
        files[filename] = file_content
        saved_files.append(filename)
        summaries.append(f"- {filename}: {result['summary']}...")
    
    # 为工具消息创建最小摘要 - 专注于收集的内容
    summary_text = f"""🔍 为 '{query}' 找到了 {len(processed_results)} 个结果：

{chr(10).join(summaries)}

文件：{', '.join(saved_files)}
💡 需要时使用 read_file() 访问完整详细信息。"""

    return Command(
        update={
            "files": files,
            "messages": [
                ToolMessage(summary_text, tool_call_id=tool_call_id)
            ],
        }
    )

@tool(parse_docstring=True)
def think_tool(reflection: str) -> str:
    """用于对研究进展和决策进行战略反思的工具。

    在每次搜索后使用此工具来系统地分析结果并规划下一步。
    这在研究工作流程中创建了一个深思熟虑的暂停，以做出高质量的决策。

    何时使用：
    - 收到搜索结果后：我发现了什么关键信息？
    - 决定下一步之前：我是否有足够的信息来全面回答？
    - 评估研究差距时：我仍然缺少什么具体信息？
    - 结束研究之前：我现在能提供完整的答案吗？
    - 问题有多复杂：我是否已达到搜索限制的数量？

    反思应该解决：
    1. 当前发现分析 - 我收集了什么具体信息？
    2. 差距评估 - 仍然缺少什么关键信息？
    3. 质量评估 - 我是否有足够的证据/示例来提供好答案？
    4. 战略决策 - 我应该继续搜索还是提供我的答案？

    Args:
        reflection: 您对研究进展、发现、差距和下一步的详细反思

    Returns:
        确认反思已记录用于决策
    """
    return f"反思已记录：{reflection}"

## 深度智能体

现在，我们可以应用我们之前学到的所有内容：

* 我们将给研究员一个 `think_tool` 和上面的 `search_tool`。
* 我们将给父智能体文件工具、一个 `think_tool` 和一个 `task` 工具。

In [ ]:
from datetime import datetime

from IPython.display import Image, display
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from utils import show_prompt, stream_agent

from deep_agents_from_scratch.file_tools import ls, read_file, write_file
from deep_agents_from_scratch.prompts import (
    FILE_USAGE_INSTRUCTIONS,
    RESEARCHER_INSTRUCTIONS,
    SUBAGENT_USAGE_INSTRUCTIONS,
    TODO_USAGE_INSTRUCTIONS,
)
from deep_agents_from_scratch.research_tools import tavily_search, think_tool, get_today_str
from deep_agents_from_scratch.state import DeepAgentState
from deep_agents_from_scratch.task_tool import _create_task_tool
from deep_agents_from_scratch.todo_tools import write_todos, read_todos

# 直接使用 create_react_agent 创建智能体
model = init_chat_model(model="anthropic:claude-sonnet-4-20250514", temperature=0.0)

# 限制
max_concurrent_research_units = 3
max_researcher_iterations = 3

# 工具
sub_agent_tools = [tavily_search, think_tool]
built_in_tools = [ls, read_file, write_file, write_todos, read_todos, think_tool]

# 创建研究子智能体
research_sub_agent = {
    "name": "research-agent",
    "description": "将研究委托给子智能体研究员。一次只给这个研究员一个主题。",
    "prompt": RESEARCHER_INSTRUCTIONS.format(date=get_today_str()),
    "tools": ["tavily_search", "think_tool"],
}

# 创建任务工具以将任务委托给子智能体
task_tool = _create_task_tool(
    sub_agent_tools, [research_sub_agent], model, DeepAgentState
)

delegation_tools = [task_tool]
all_tools = sub_agent_tools + built_in_tools + delegation_tools  # 搜索可用于主智能体的琐碎情况

# 构建提示
SUBAGENT_INSTRUCTIONS = SUBAGENT_USAGE_INSTRUCTIONS.format(
    max_concurrent_research_units=max_concurrent_research_units,
    max_researcher_iterations=max_researcher_iterations,
    date=datetime.now().strftime("%a %b %-d, %Y"),
)

In [ ]:
show_prompt(RESEARCHER_INSTRUCTIONS)

In [ ]:
INSTRUCTIONS = (
    "# TODO 管理\n"
    + TODO_USAGE_INSTRUCTIONS
    + "\n\n"
    + "=" * 80
    + "\n\n"
    + "# 文件系统使用\n"
    + FILE_USAGE_INSTRUCTIONS
    + "\n\n"
    + "=" * 80
    + "\n\n"
    + "# 子智能体委托\n"
    + SUBAGENT_INSTRUCTIONS
)

show_prompt(INSTRUCTIONS)

In [ ]:
# 创建智能体
agent = create_react_agent(
    model, all_tools, prompt=INSTRUCTIONS, state_schema=DeepAgentState
)

# 显示智能体
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
from utils import format_messages

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "给我一个模型上下文协议（MCP）的概述。",
            }
        ],
    }
)

format_messages(result["messages"])

跟踪：
https://smith.langchain.com/public/3a389ec6-8e6e-4f9e-9a82-0d0a9569e6f8/r
<!-- https://smith.langchain.com/public/1df7a10e-1465-499c-a3e0-86c1d5429324/r -->

## 使用深度智能体包

现在您了解了底层模式！

您可以[使用 `deepagents` 包](https://github.com/hwchase17/deepagents)作为简单抽象：

* 它包含文件系统工具
* 它包含 todo 工具
* 它包含任务工具

您只需要提供子智能体和您希望子智能体使用的任何工具。

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    sub_agent_tools,
    INSTRUCTIONS,
    subagents=[research_sub_agent],
    model=model,
)

# 显示智能体
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "给我一个模型上下文协议（MCP）的非常简短的概述。",
            }
        ],
    }
)

format_messages(result["messages"])

跟踪：
https://smith.langchain.com/public/1d626d81-a102-4588-a2fb-cab40a7271f1/r
<!-- https://smith.langchain.com/public/1ae2d7f6-f901-4ebd-b6c3-6657a55f88ae/r -->